# Hyperparameter Tuning with Keras Tuner (Optional)

This notebook is optional: it extends notebook 02.

In notebook 02 you chose the head of the transfer-learning model by hand: no hidden layer, a dropout rate of 0.2 and a learning rate of 0.001. A network has many such hyperparameters, and trying combinations one by one takes a long time. For scikit-learn models you used `GridSearchCV` and `RandomizedSearchCV` for this. [**Keras Tuner**](https://keras.io/keras_tuner/) does the same for Keras models: you describe the values each hyperparameter may take, and the tuner builds, trains and compares the models for you.

## Learning Objectives

By the end of this notebook, you should be able to:

- Define a search space for the hyperparameters of a Keras model with a `HyperParameters` object.
- Run a random search with Keras Tuner and read its results.
- Rebuild and train the model with the best hyperparameters and compare it with a manually chosen model on the validation set.
- Choose between random search and Hyperband for a tuning task.

## Setup

In [ ]:
from pathlib import Path

import keras
import keras_tuner as kt
import numpy as np
import pandas as pd
from keras.applications import mobilenet_v2

keras.utils.set_random_seed(42)
IMAGE_SIZE = (160, 160)

## 1. The Features of Notebook 02

Tuning trains many models, so each one should train quickly. As in notebook 02, the frozen MobileNetV2 base turns every bean image into a feature vector once; the tuner then only trains small heads on these vectors, which takes a few seconds per model. The images are the same as in notebook 02, from `data/beans/`.

In [ ]:
BEANS = Path("data") / "beans"
folders = {split: BEANS / split for split in ["train", "validation"]}

base = mobilenet_v2.MobileNetV2(
    include_top=False, weights="imagenet", input_shape=IMAGE_SIZE + (3,), pooling="avg"
)


def extract_features(split):
    """Return the feature vectors and labels of one split, from an unshuffled dataset."""
    dataset = keras.utils.image_dataset_from_directory(
        folders[split], image_size=IMAGE_SIZE, batch_size=32, label_mode="int", shuffle=False
    )
    features = base.predict(dataset.map(lambda x, y: mobilenet_v2.preprocess_input(x)), verbose=0)
    labels = np.concatenate([y.numpy() for _, y in dataset])
    return features, labels


X_train, y_train = extract_features("train")
X_valid, y_valid = extract_features("validation")
print("Training features:", X_train.shape, " validation features:", X_valid.shape)

## 2. A Manually Chosen Head

As a reference, train a head with one hidden layer of 128 neurons, a dropout rate of 0.3 and a learning rate of 0.001, values that are a reasonable first guess. [`EarlyStopping`](https://keras.io/api/callbacks/early_stopping/) on the validation loss keeps the run short and restores the best epoch.

In [ ]:
def build_head(units, dropout_rate, learning_rate):
    """Return a compiled head for the 1,280 features: one hidden layer, dropout and a softmax output."""
    model = keras.Sequential(
        [
            keras.Input(shape=(X_train.shape[1],)),
            keras.layers.Dense(units=units, activation="relu"),
            keras.layers.Dropout(dropout_rate),
            keras.layers.Dense(units=3, activation="softmax"),
        ]
    )
    model.compile(
        loss="sparse_categorical_crossentropy",
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        metrics=["accuracy"],
    )
    return model


def early_stopping():
    return keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)


keras.utils.set_random_seed(42)
manual_head = build_head(units=128, dropout_rate=0.3, learning_rate=1e-3)
manual_head.fit(X_train, y_train, validation_data=(X_valid, y_valid), epochs=50, callbacks=[early_stopping()], verbose=0)
manual_accuracy = manual_head.evaluate(X_valid, y_valid, verbose=0)[1]
print(f"Manually chosen head, validation accuracy: {manual_accuracy:.3f}")

## 3. Describe the Search Space

Keras Tuner needs a function that builds a model from a [`HyperParameters`](https://keras.io/api/keras_tuner/hyperparameters/) object, usually called `hp`. Instead of a fixed number, you ask `hp` for a value and describe the values it may take:

| Method | Values | Example below |
|---|---|---|
| `hp.Int(name, min_value, max_value, step)` | whole numbers in a range | number of neurons: 32 to 256 in steps of 32 |
| `hp.Float(name, min_value, max_value, step)` | decimal numbers in a range | dropout rate: 0.0 to 0.5 in steps of 0.1 |
| `hp.Choice(name, values)` | one value from a list | learning rate: 0.01, 0.001 or 0.0001 |

The function below reuses `build_head` and only replaces the fixed values. These ranges contain $8 \cdot 6 \cdot 3 = 144$ combinations.

In [ ]:
def build_tunable_head(hp):
    """Return a head whose hyperparameters are chosen by the tuner."""
    return build_head(
        units=hp.Int("units", min_value=32, max_value=256, step=32),
        dropout_rate=hp.Float("dropout_rate", min_value=0.0, max_value=0.5, step=0.1),
        learning_rate=hp.Choice("learning_rate", values=[1e-2, 1e-3, 1e-4]),
    )

## 4. Run a Random Search

[`RandomSearch`](https://keras.io/api/keras_tuner/tuners/random/) works like `RandomizedSearchCV`: it draws `max_trials` combinations at random, trains a model for each and keeps the one with the best `objective`, here the validation accuracy. Unlike `RandomizedSearchCV`, it scores every combination on one fixed validation set instead of with cross-validation. Its arguments:

- `objective="val_accuracy"`: the value to maximise.
- `max_trials=15`: try 15 of the 144 combinations.
- `seed=42`: the same combinations on every run.
- `directory` and `project_name`: the folder where the tuner stores every trial, so a search can be resumed. `overwrite=True` starts a new search each time you run the cell.

`tuner.search()` takes the same arguments as `fit()`. Each trial uses early stopping, so it ends once the validation loss stops improving.

In [ ]:
tuner = kt.RandomSearch(
    build_tunable_head,
    objective="val_accuracy",
    max_trials=15,
    seed=42,
    directory="tuner",
    project_name="beans_head",
    overwrite=True,
)
tuner.search_space_summary()
tuner.search(X_train, y_train, validation_data=(X_valid, y_valid), epochs=50, callbacks=[early_stopping()], verbose=0)

The tuner keeps the record of all trials in its [oracle](https://keras.io/api/keras_tuner/oracles/), the part that decides which combination to try next. `tuner.oracle.get_best_trials()` returns the trials sorted by their score, the best validation accuracy reached in any epoch of the trial. The table shows the hyperparameters and the validation accuracy of all 15.

In [ ]:
trials = [{**trial.hyperparameters.values, "val_accuracy": trial.score} for trial in tuner.oracle.get_best_trials(15)]
pd.DataFrame(trials).round(4)

The 15 trials reach validation accuracies between 0.910 and 0.940. Three trials share the best score of 0.940, with very different settings: 96 neurons, a dropout rate of 0.4 and a learning rate of 0.001; 192 neurons, 0.4 and 0.01; and 32 neurons, 0.1 and 0.0001. Every learning rate appears both near the top and near the bottom of the table, so no hyperparameter shows a clear effect. On 133 validation images one image is 0.0075, so the whole table spans only four images: on these features, many heads work about equally well.

## 5. Train the Best Model

`get_best_hyperparameters()` returns the best combination. `tuner.hypermodel.build()` builds a new model with it, which you train as usual.

In [ ]:
best_hp = tuner.get_best_hyperparameters(1)[0]
print("Best hyperparameters:", best_hp.values)

keras.utils.set_random_seed(42)
best_head = tuner.hypermodel.build(best_hp)
best_head.fit(X_train, y_train, validation_data=(X_valid, y_valid), epochs=50, callbacks=[early_stopping()], verbose=0)
best_accuracy = best_head.evaluate(X_valid, y_valid, verbose=0)[1]
print(f"Tuned head: {best_accuracy:.3f}, manually chosen head: {manual_accuracy:.3f}")

Trained again, the best combination reaches 0.917, less than the 0.940 of its trial and exactly the same as the manually chosen head. This is not an error but two effects you should expect. First, the tuner records the best validation accuracy of any epoch of a trial, while early stopping restores the epoch with the lowest validation loss. Second, the new run starts from different random weights, and the tuner reported the highest of 15 runs: on 133 images, the difference between 0.940 and 0.917 is 3 images. The search confirms that the manual guess was already good; it is not a reliable estimate of the final accuracy. More validation data, cross-validation or averaging several runs per trial (`executions_per_trial`) make the comparison more reliable.

## 6. Random Search or Hyperband?

Random search trains every trial until early stopping ends it. [**Hyperband**](https://keras.io/api/keras_tuner/tuners/hyperband/) saves time by training many combinations for only a few epochs first, keeping the most promising third and training those longer, and repeating this until a few remain. Weak combinations are stopped early, so most of the computing time goes to the promising ones. Because a combination that starts slowly can be stopped too early, Hyperband runs this procedure several times with different numbers of starting epochs (its **brackets**). Creating it looks almost the same:

```python
tuner = kt.Hyperband(build_tunable_head, objective="val_accuracy", max_epochs=50, factor=3, seed=42,
                     directory="tuner", project_name="beans_hyperband", overwrite=True)
```

| | Random search | Hyperband |
|---|---|---|
| Trials | a fixed number, each trained until early stopping ends it | many, most stopped after a few epochs |
| Best when | each trial is fast, as on extracted features | each trial is slow, as when the base runs inside the model |
| Risk | can miss good regions of the search space | can discard combinations that start slowly but end well |

**Your turn:** the flower classification project trains the full transfer-learning model, with augmentation and the base inside, which takes about ten seconds per epoch. Would you use random search or Hyperband there, and why?

<details>
<summary>Show the answer</summary>

Hyperband. Each trial is slow because the base runs on every image in every epoch, so stopping weak combinations after a few epochs saves most of the computing time. Random search would train every combination until early stopping, which could take hours. On extracted features, where a trial takes seconds, random search is simpler and the time saved by Hyperband is small.

</details>

Two cautions apply to both. First, the tuner chose the best of 15 heads by their validation accuracy, so that accuracy is slightly optimistic, for the same reason as in model selection: the test set gives the unbiased estimate, once, for the final model. Second, with only 133 validation images, one image is 0.75 percentage points; differences of one or two images between trials are not reliable.

## Try It: Tune the Depth of the Head (Optional)

Add a hyperparameter for the **number of hidden layers**, 1 or 2, to the search space. Complete `build_deeper_head` so that it adds the hidden layers in a loop, each with its own number of neurons, and run a random search with 15 trials in a new project. Section 5 showed that the best trial score is optimistic, so retrain the best deeper head and compare its validation accuracy with the retrained head of section 5.

In [ ]:
def build_deeper_head(hp):
    """Return a head with 1 or 2 hidden layers chosen by the tuner."""
    layers = [keras.Input(shape=(X_train.shape[1],))]
    # TODO: ask hp for the number of hidden layers, a whole number from 1 to 2
    n_layers = ...
    for i in range(n_layers):
        # TODO: a Dense layer with ReLU whose units come from hp.Int(f"units_{i}", 32, 256, step=32)
        layers.append(...)
        layers.append(keras.layers.Dropout(hp.Float("dropout_rate", 0.0, 0.5, step=0.1)))
    layers.append(keras.layers.Dense(units=3, activation="softmax"))
    model = keras.Sequential(layers)
    model.compile(
        loss="sparse_categorical_crossentropy",
        optimizer=keras.optimizers.Adam(learning_rate=hp.Choice("learning_rate", values=[1e-2, 1e-3, 1e-4])),
        metrics=["accuracy"],
    )
    return model


# TODO: a RandomSearch tuner for build_deeper_head with the same settings as in section 4 and project_name="beans_deeper"
deeper_tuner = ...
deeper_tuner.search(X_train, y_train, validation_data=(X_valid, y_valid), epochs=50, callbacks=[early_stopping()], verbose=0)
deeper_hp = deeper_tuner.get_best_hyperparameters(1)[0]
print("Best hyperparameters:", deeper_hp.values)

keras.utils.set_random_seed(42)
deeper_head = deeper_tuner.hypermodel.build(deeper_hp)
deeper_head.fit(X_train, y_train, validation_data=(X_valid, y_valid), epochs=50, callbacks=[early_stopping()], verbose=0)
print(f"Retrained deeper head: {deeper_head.evaluate(X_valid, y_valid, verbose=0)[1]:.3f} "
      f"(one hidden layer, retrained: {best_accuracy:.3f})")

## Summary

- Keras Tuner searches the hyperparameters of a Keras model, scoring every combination on a validation set. A model-building function asks a `HyperParameters` object for each value with `hp.Int`, `hp.Float` or `hp.Choice`.
- `RandomSearch` trains a fixed number of random combinations in full; Hyperband trains many combinations briefly and continues only the promising ones.
- Tuning on extracted features makes each trial take seconds instead of minutes.
- The best validation score of a search is slightly optimistic, and with a small validation set, small differences between trials are not reliable.

As a further exercise, tune the head of the sea animals in [03 - Exercise: Classify Sea Animals with Transfer Learning](03_exercise_sea_animals.ipynb) in the same way.

## References

- Keras Tuner: [Getting started](https://keras.io/keras_tuner/getting_started/), [`RandomSearch`](https://keras.io/api/keras_tuner/tuners/random/) and [`Hyperband`](https://keras.io/api/keras_tuner/tuners/hyperband/).
- Li, L., Jamieson, K., DeSalvo, G., Rostamizadeh, A. and Talwalkar, A. (2018). [Hyperband: A Novel Bandit-Based Approach to Hyperparameter Optimization](https://jmlr.org/papers/v18/16-558.html). *Journal of Machine Learning Research*, 18.
- Bergstra, J. and Bengio, Y. (2012). [Random Search for Hyper-Parameter Optimization](https://jmlr.org/papers/v13/bergstra12a.html). *Journal of Machine Learning Research*, 13: why random search often beats grid search.